# 02 -- Phase 1: Calibration (ISR)

Instrument Signature Removal: build master bias/dark/flat, subtract/divide
them from each science frame, reject cosmic rays, convert ADU to electrons,
and **seed the error budget** into the ERR plane while flagging DQ.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG  --  EDIT THESE PATHS FOR YOUR ENVIRONMENT
# ============================================================
import os

# 1) Shared, read-only Astrometry.net index directory (used by Phase 2):
os.environ["CASSA_ASTROMETRY_INDEX"] = os.path.abspath("../../astrometry_data")

# 2) The provided workshop dataset + a writable work directory:
RAW_DIR   = "../raw"     # provided raw frames, one level up from notebooks/
WORK_DIR  = "../work"    # writable output dir, one level up from notebooks/

RAW_DIR   = os.path.abspath(os.path.expandvars(RAW_DIR))
WORK_DIR  = os.path.abspath(os.path.expandvars(WORK_DIR))

# These are relative to the working directory, which Jupyter sets to this
# notebook's folder. Fail loudly here rather than confusingly further down.
assert os.path.isdir(RAW_DIR), (
    f"RAW_DIR not found: {RAW_DIR}\nRun this notebook from the notebooks/ "
    f"directory, or set RAW_DIR/WORK_DIR to absolute paths above."
)

# Each phase writes into its own directory under WORK_DIR.
PHASE1_DIR = os.path.join(WORK_DIR, "phase1")   # calibrated frames
PHASE2_DIR = os.path.join(WORK_DIR, "phase2")   # master stacks + WCS
PHASE3_DIR = os.path.join(WORK_DIR, "phase3")   # flux-calibrated + catalogs
PHASE4_DIR = os.path.join(WORK_DIR, "phase4")   # diagnostics report
os.makedirs(WORK_DIR, exist_ok=True)
print("RAW_DIR    =", RAW_DIR)
print("PHASE1_DIR =", PHASE1_DIR)
print("PHASE2_DIR =", PHASE2_DIR)

## Run Phase 1
Reads raw frames from `RAW_DIR`, writes `calibrated_*.fits` (SCI/ERR/DQ) into `PHASE1_DIR`.

In [ ]:
from cassa_photometry.config import load_config
from cassa_photometry.phase1_calibration import run as run_p1
cfg = load_config()
run_p1(RAW_DIR, PHASE1_DIR, config=cfg)

## Inspect a calibrated frame
Note the ERR plane is now populated and DQ carries saturation / CR / bad-pixel flags.

In [ ]:
import glob, os, numpy as np
from cassa_photometry.fits_utils import read_mef, DQ_FLAG_NAMES
out = sorted(glob.glob(os.path.join(PHASE1_DIR, 'calibrated_*.fits')))
print(len(out), 'calibrated frames')
sci, err, dq, hdr = read_mef(out[0])
print('BUNIT:', hdr.get('BUNIT'))
print('median SCI (e-):', float(np.nanmedian(sci)))
print('median ERR (e-):', float(np.nanmedian(err)))
for flag, name in DQ_FLAG_NAMES.items():
    print(f'  {name:>11}:', int(np.count_nonzero(dq & flag)), 'px')

### Exercise 1 -- what did calibration actually change?
Compare the same field before and after Phase 1: load the raw counterpart of
the calibrated frame above (`X.fits` in `RAW_DIR` <-> `calibrated_X.fits` in
`PHASE1_DIR`). Raw is in ADU and calibrated is in electrons, so convert the raw
frame to electrons with the same gain Phase 1 uses (no need to touch the FITS
file itself) before comparing -- background level (bias + dark removed) and
the fact that only the calibrated frame carries real `ERR`/`DQ` planes.

In [ ]:
import os
import numpy as np
from astropy.io import fits
from cassa_photometry.instruments import ITelescopeNetworkProfile

raw_name = os.path.basename(out[0]).replace('calibrated_', '', 1)
raw_path = os.path.join(RAW_DIR, raw_name)

with fits.open(raw_path) as hdul:
    raw_header = hdul[0].header
    raw_adu = hdul[0].data.astype(np.float64)
    n_ext_raw = len(hdul)

# Same ADU -> electrons conversion Phase 1 applies, so raw and calibrated are
# on the same footing (nothing is written back to the FITS file).
instrument = ITelescopeNetworkProfile()
gain = instrument.get_gain(raw_header)
raw_e = raw_adu * gain

def stats(a):
    return dict(median=np.nanmedian(a), std=np.nanstd(a), min=np.nanmin(a), max=np.nanmax(a))

raw_stats = stats(raw_e)
sci_stats = stats(sci)

print(f"Raw file       : {raw_name}")
print(f"Calibrated file: {os.path.basename(out[0])}")
print(f"Gain applied to raw for comparison: {gain} e-/ADU\n")

print(f"{'':>10} {'RAW (e-)':>14} {'CALIBRATED (e-)':>18}")
for key in ('median', 'std', 'min', 'max'):
    print(f"{key:>10} {raw_stats[key]:14.2f} {sci_stats[key]:18.2f}")

print(f"\nMedian drop (raw - calibrated): {raw_stats['median'] - sci_stats['median']:.2f} e-"
      f"  (bias + dark subtracted out by Phase 1)")
print("Calibrated min is negative: once the bias/dark pedestal is removed,")
print("pure background noise fluctuates around zero.")

print(f"\nHDU count -- raw: {n_ext_raw} (SCI only)   calibrated: 3 (SCI/ERR/DQ)")
print(f"Calibrated ERR available: {err is not None}   DQ available: {dq is not None}")


### Exercise 2 -- where did the cosmic rays go?
The `DQ` plane records every pixel `astroscrappy` rejected. Pull out the
`COSMIC_RAY` pixels, then look at those same pixel positions in the *raw*
frame. How far above the sky were they before, and what are they now? Print
the five most extreme ones side by side.

In [ ]:
from cassa_photometry.fits_utils import DQ_COSMIC_RAY, DQ_SATURATED

# Pixels Phase 1 flagged as cosmic rays, and what they looked like in the raw frame.
cr = (dq & DQ_COSMIC_RAY) > 0
print(f"{int(cr.sum())} pixels flagged COSMIC_RAY\n")

sky = np.nanmedian(raw_e)
print(f"Raw sky level (median)            : {sky:8.1f} e-")
print(f"Raw value at CR pixels  (median)  : {np.nanmedian(raw_e[cr]):8.1f} e-  "
      f"({np.nanmedian(raw_e[cr]) / sky:.1f}x the sky)")
print(f"Raw value at CR pixels  (max)     : {np.nanmax(raw_e[cr]):8.1f} e-")
print(f"Calibrated value at those pixels  : {np.nanmedian(sci[cr]):8.1f} e-  (median)")

# The 5 most extreme cosmic rays, before and after.
order = np.argsort(raw_e[cr])[::-1][:5]
ys, xs = np.where(cr)
print(f"\n{'(y, x)':>14} {'raw (e-)':>12} {'calibrated (e-)':>17}")
for k in order:
    y, x = ys[k], xs[k]
    print(f"{f'({y}, {x})':>14} {raw_e[y, x]:12.1f} {sci[y, x]:17.1f}")

print("\nastroscrappy detected these sharp spikes and replaced them, and the DQ")
print("plane records where -- nothing is silently thrown away.")


### Exercise 3 -- is the ERR plane just Poisson + read noise?
Phase 1 "seeds the error budget". Test how complete that budget is: compare
the actual `ERR` plane against $\sqrt{S + \mathrm{RN}^2}$, the error you would
predict if the science frame were the only noise source. They do **not**
match -- work out how much extra noise there is, and where it comes from.

In [ ]:
from cassa_photometry.instruments import ITelescopeNetworkProfile

# What would the error be if the science frame were the ONLY noise source?
read_noise = ITelescopeNetworkProfile().get_read_noise(raw_header)  # e-
good = (dq == 0) & np.isfinite(sci) & np.isfinite(err)

predicted = np.sqrt(np.clip(sci[good], 0, None) + read_noise ** 2)  # Poisson + read noise
actual = err[good]

print(f"Read noise from the instrument profile : {read_noise} e-")
print(f"\n{'':>34} {'median (e-)':>12}")
print(f"{'Actual ERR plane':>34} {np.median(actual):12.2f}")
print(f"{'Poisson + read noise alone':>34} {np.median(predicted):12.2f}")

# Whatever is left over was contributed by the master bias/dark/flat.
excess = np.sqrt(np.clip(np.median(actual) ** 2 - np.median(predicted) ** 2, 0, None))
print(f"{'Extra noise from calibration':>34} {excess:12.2f}   (added in quadrature)")

print(f"\nRatio actual/predicted (median): {np.median(actual) / np.median(predicted):.2f}x")
print("\nThe ERR plane is LARGER than the naive sqrt(S + RN^2): subtracting the")
print("master bias and dark, and dividing by the flat, each fold their own")
print("uncertainty into the budget. ccdproc propagates all of it for you --")
print("that is why the masters are built with an uncertainty in the first place.")
